<a href="https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/XGBoost2WWSantanderHyperparameterSweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# xgboost2ww Santander Customer Transaction Hyperparameter Sweep (Underfit → Tuned → Mild Overfit)

This notebook demonstrates an **industry-style sweep** on Santander Customer Transaction data using **XGBoost + xgboost2ww + WeightWatcher**.

## Dataset source and loading path
- Dataset zip is expected at: `/content/drive/MyDrive/xgboost2ww_runs/sandata.zip`
- Zip is extracted to: `/content/sandata`
- Training file used for modeling: `/content/sandata/train.csv`

## Reproducibility protocol
- Fixed seeds for NumPy / train-test split / XGBoost random state.
- Stratified split: `test_size=0.20`, `random_state=0`.

## Sweep design objective
- Intentionally span **underfit → tuned → mild overfit** regimes.
- Compare predictive quality against WeightWatcher diagnostics.

## xgboost2ww conversion settings
- `convert(bst, X_train, y_train, W="W7", return_type="torch", nfolds=5, t_points=40, random_state=0, train_params=..., num_boost_round=...)`
- Important: conversion always includes `X_train` and `y_train` positional arguments.

## WeightWatcher settings
- Run with `randomize=True` for all configs.
- `alpha` (heavy-tail exponent) is used as primary spectral indicator.
- `rand_num_spikes` is used as trap/instability signal.
- `detX=True` is computed for three representative configs (best / underfit / overfit exemplars).

## Expected qualitative pattern
- Best generalizing models often trend toward `alpha ≈ 2`.
- Underfit models often have larger alpha with weaker accuracy.
- Overfit/unstable models may show `alpha < 2` and/or higher trap signals.
- Exact scores vary by run; **the qualitative relationship is the main result**.


## Google Drive checkpoint locations (duplicated here for run notes)
- Root: `/content/drive/MyDrive/xgboost2ww_runs`
- Results CSV: `santander_hyperparameter_sweep_results.csv`
- Plots: `santander_acc_vs_alpha_test.png`, `santander_acc_vs_alpha_train.png`, `santander_test_vs_spikes.png`


In [ ]:
from pathlib import Path
import os
import zipfile

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CHECKPOINT_ROOT_COLAB = Path('/content/drive/MyDrive/xgboost2ww_runs')
CHECKPOINT_ROOT_LOCAL = Path('./checkpoints')
EXPERIMENT_NAME = 'santander_hparam_sweep'
OUTPUT_FILENAME = 'santander_hyperparameter_sweep_results.csv'

if IN_COLAB:
    drive.mount('/content/drive')
    CHECKPOINT_ROOT = CHECKPOINT_ROOT_COLAB
else:
    CHECKPOINT_ROOT = CHECKPOINT_ROOT_LOCAL

CHECKPOINT_DIR = CHECKPOINT_ROOT / EXPERIMENT_NAME
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_RESULTS_CSV = CHECKPOINT_DIR / OUTPUT_FILENAME

DRIVE_ZIP_PATH = Path('/content/drive/MyDrive/xgboost2ww_runs/sandata.zip')
LOCAL_DATA_DIR = Path('/content/sandata')
LOCAL_TRAIN_CSV = LOCAL_DATA_DIR / 'train.csv'

print('IN_COLAB:', IN_COLAB)
print('Checkpoint directory:', CHECKPOINT_DIR)
print('Checkpoint CSV:', CHECKPOINT_RESULTS_CSV)
print('Expected zip path:', DRIVE_ZIP_PATH)
print('Local extract dir:', LOCAL_DATA_DIR)


In [ ]:
# Optional install cell (useful in Colab)
# !pip -q install xgboost weightwatcher xgboost2ww


## Imports and global settings


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

import xgboost as xgb
import weightwatcher as ww
from xgboost2ww import convert


## Reproducibility + experiment toggles


In [ ]:
RNG = 0
random.seed(RNG)
np.random.seed(RNG)
TEST_SIZE = 0.20
VALID_SIZE_FROM_TRAIN = 0.20
NFOLDS = 5
T_POINTS = 40
W_MATRIX = "W7"
DET_X_SWEEP = False
DET_X_KEY_MODELS = True

BEST_NUM_BOOST_ROUND = 6000
BEST_EARLY_STOP = 150


## Load Santander dataset from Google Drive zip into fast local storage


In [ ]:
if not DRIVE_ZIP_PATH.exists():
    raise FileNotFoundError(f"sandata.zip not found at: {DRIVE_ZIP_PATH}")

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
    zf.extractall(LOCAL_DATA_DIR)

if not LOCAL_TRAIN_CSV.exists():
    nested = list(LOCAL_DATA_DIR.rglob('train.csv'))
    if nested:
        LOCAL_TRAIN_CSV = nested[0]

if not LOCAL_TRAIN_CSV.exists():
    raise FileNotFoundError(f"train.csv not found after unzip under: {LOCAL_DATA_DIR}")

print('Using train.csv at:', LOCAL_TRAIN_CSV)

df = pd.read_csv(LOCAL_TRAIN_CSV)
required_cols = {'ID_code', 'target'}
if not required_cols.issubset(df.columns):
    raise ValueError(f"Missing required columns: {required_cols - set(df.columns)}")

X_df = df.drop(columns=['ID_code', 'target'])
y = df['target'].astype(int).to_numpy()

nan_count_before = int(X_df.isna().sum().sum())
if nan_count_before > 0:
    X_df = X_df.fillna(X_df.median(numeric_only=True))

X = X_df.to_numpy(dtype=np.float32)

print('X.shape:', X.shape, 'y.shape:', y.shape)
print('Class balance (positive rate):', float(y.mean()))
print('NaN count before imputation:', nan_count_before)
print('NaNs after imputation:', int(np.isnan(X).sum()))


## Stratified split + sanity checks


In [ ]:
idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=TEST_SIZE, random_state=RNG, stratify=y
)
Xtr, Xte = X[idx_train], X[idx_test]
ytr, yte = y[idx_train], y[idx_test]

print('Train/Test sizes:', Xtr.shape[0], Xte.shape[0])
print('Train class balance:', float(ytr.mean()))
print('Test class balance:', float(yte.mean()))
print('No NaNs in train/test:', (not np.isnan(Xtr).any()) and (not np.isnan(Xte).any()))


## Training/evaluation helpers + xgboost2ww/WeightWatcher extraction


In [ ]:
BASE_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": ["logloss", "auc"],
    "tree_method": "hist",
    "seed": RNG,
}


def _to_numeric_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            vals = pd.to_numeric(df[c], errors="coerce")
            if vals.notna().any():
                return float(vals.dropna().iloc[0])
    return np.nan


def _extract_ww_metrics(details_df):
    alpha = np.nan
    spikes = np.nan
    detx_val = np.nan
    if isinstance(details_df, pd.DataFrame) and len(details_df) > 0:
        alpha = _to_numeric_first(details_df, [
            "alpha", "alpha_weighted", "alpha_hat", "PL_alpha", "mp_softrank_alpha"
        ])
        spikes = _to_numeric_first(details_df, [
            "rand_num_spikes", "num_rand_spikes", "rand_spikes", "num_spikes", "num_traps"
        ])
        detx_val = _to_numeric_first(details_df, [
            "detX_val", "detX", "det_x", "log_detX"
        ])
    return alpha, spikes, detx_val


def train_eval_one(cfg, config_id, detx=False):
    params = BASE_PARAMS.copy()
    params.update({
        "max_depth": int(cfg["max_depth"]),
        "eta": float(cfg["eta"]),
        "subsample": float(cfg["subsample"]),
        "colsample_bytree": float(cfg["colsample_bytree"]),
        "min_child_weight": float(cfg["min_child_weight"]),
        "gamma": float(cfg["gamma"]),
        "reg_alpha": float(cfg["reg_alpha"]),
        "reg_lambda": float(cfg["reg_lambda"]),
    })

    num_boost_round = int(cfg["num_boost_round"])
    early_stop = cfg.get("early_stopping_rounds", None)

    x_train, x_val, y_train, y_val = train_test_split(
        Xtr, ytr, test_size=VALID_SIZE_FROM_TRAIN, random_state=RNG, stratify=ytr
    )
    dtr = xgb.DMatrix(x_train, label=y_train)
    dval = xgb.DMatrix(x_val, label=y_val)
    dtr_all = xgb.DMatrix(Xtr, label=ytr)
    dte = xgb.DMatrix(Xte, label=yte)

    watchlist = [(dtr, "train"), (dval, "valid")]
    train_kwargs = dict(params=params, dtrain=dtr, num_boost_round=num_boost_round, evals=watchlist, verbose_eval=False)
    if early_stop is not None:
        train_kwargs["early_stopping_rounds"] = int(early_stop)

    row = {
        "config_id": config_id,
        "group": cfg["group"],
        "error": "",
        "max_depth": int(cfg["max_depth"]),
        "eta": float(cfg["eta"]),
        "subsample": float(cfg["subsample"]),
        "colsample_bytree": float(cfg["colsample_bytree"]),
        "min_child_weight": float(cfg["min_child_weight"]),
        "gamma": float(cfg["gamma"]),
        "reg_alpha": float(cfg["reg_alpha"]),
        "reg_lambda": float(cfg["reg_lambda"]),
        "num_boost_round": num_boost_round,
        "early_stopping_rounds": np.nan if early_stop is None else int(early_stop),
    }

    try:
        bst = xgb.train(**train_kwargs)
        best_iteration = int(getattr(bst, "best_iteration", num_boost_round - 1))

        p_tr = (bst.predict(dtr_all) >= 0.5).astype(int)
        p_te_prob = bst.predict(dte)
        p_te = (p_te_prob >= 0.5).astype(int)

        row.update({
            "best_iteration": best_iteration,
            "train_accuracy": float(accuracy_score(ytr, p_tr)),
            "test_accuracy": float(accuracy_score(yte, p_te)),
            "test_auc": float(roc_auc_score(yte, p_te_prob)),
            "test_logloss": float(log_loss(yte, np.clip(p_te_prob, 1e-6, 1-1e-6))),
        })

        ww_layer = convert(
            bst,
            Xtr,
            ytr,
            nfolds=NFOLDS,
            t_points=T_POINTS,
            random_state=RNG,
            W=W_MATRIX,
            return_type="torch",
            train_params=params,
            num_boost_round=(best_iteration + 1),
        )

        ww_linear = ww_layer[0] if hasattr(ww_layer, "__getitem__") and len(ww_layer) > 0 else ww_layer
        if hasattr(ww_linear, "weight"):
            w_shape = tuple(int(v) for v in ww_linear.weight.shape)
            ww_max_n = int(np.prod(w_shape))
        else:
            ww_max_n = int(Xtr.shape[0] * T_POINTS)

        watcher = ww.WeightWatcher(model=ww_layer)
        details_df = watcher.analyze(randomize=True, plot=False, detX=bool(detx), max_N=ww_max_n)
        alpha, spikes, detx_val = _extract_ww_metrics(details_df)
        row.update({"alpha": alpha, "rand_num_spikes": spikes, "detX_val": detx_val, "ww_max_n": ww_max_n})
    except Exception as e:
        row.update({
            "best_iteration": np.nan,
            "train_accuracy": np.nan,
            "test_accuracy": np.nan,
            "test_auc": np.nan,
            "test_logloss": np.nan,
            "alpha": np.nan,
            "rand_num_spikes": np.nan,
            "detX_val": np.nan,
            "ww_max_n": np.nan,
            "error": f"{type(e).__name__}: {e}",
        })
    return row


## Best Model section (strong baseline with early stopping)


In [ ]:
best_cfg = {
    "group": "best",
    "max_depth": 6,
    "eta": 0.03,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "min_child_weight": 2,
    "gamma": 0.5,
    "reg_alpha": 1.0,
    "reg_lambda": 8.0,
    "num_boost_round": BEST_NUM_BOOST_ROUND,
    "early_stopping_rounds": BEST_EARLY_STOP,
}
best_row = train_eval_one(best_cfg, config_id="BEST", detx=True)
print('Best Model hyperparameters:', best_cfg)
print('best_iteration:', best_row.get('best_iteration'))
print('train_accuracy:', best_row.get('train_accuracy'))
print('test_accuracy:', best_row.get('test_accuracy'))
print('test_auc:', best_row.get('test_auc'))
print('test_logloss:', best_row.get('test_logloss'))
print('alpha:', best_row.get('alpha'), 'rand_num_spikes:', best_row.get('rand_num_spikes'))


## Hyperparameter sweep: underfit / tuned / overfit-leaning


In [ ]:
underfit_configs = [
    dict(group="underfit", max_depth=1, eta=0.30, num_boost_round=150, min_child_weight=30, gamma=10, subsample=0.60, colsample_bytree=0.60, reg_lambda=120, reg_alpha=2.0, early_stopping_rounds=30),
    dict(group="underfit", max_depth=2, eta=0.25, num_boost_round=200, min_child_weight=20, gamma=8, subsample=0.70, colsample_bytree=0.70, reg_lambda=80, reg_alpha=1.5, early_stopping_rounds=30),
    dict(group="underfit", max_depth=3, eta=0.20, num_boost_round=300, min_child_weight=15, gamma=5, subsample=0.75, colsample_bytree=0.75, reg_lambda=40, reg_alpha=1.0, early_stopping_rounds=40),
    dict(group="underfit", max_depth=2, eta=0.10, num_boost_round=400, min_child_weight=50, gamma=20, subsample=0.50, colsample_bytree=0.50, reg_lambda=200, reg_alpha=2.0, early_stopping_rounds=40),
]

tuned_configs = [
    dict(group="tuned", max_depth=4, eta=0.08, num_boost_round=3000, min_child_weight=2, gamma=0.5, subsample=0.80, colsample_bytree=0.80, reg_lambda=10, reg_alpha=0.5, early_stopping_rounds=120),
    dict(group="tuned", max_depth=5, eta=0.05, num_boost_round=5000, min_child_weight=2, gamma=0.3, subsample=0.85, colsample_bytree=0.90, reg_lambda=8, reg_alpha=1.0, early_stopping_rounds=150),
    dict(group="tuned", max_depth=6, eta=0.03, num_boost_round=7000, min_child_weight=3, gamma=0.5, subsample=0.90, colsample_bytree=0.85, reg_lambda=12, reg_alpha=0.2, early_stopping_rounds=150),
    dict(group="tuned", max_depth=7, eta=0.04, num_boost_round=4500, min_child_weight=1, gamma=1.0, subsample=0.75, colsample_bytree=0.75, reg_lambda=5, reg_alpha=0.0, early_stopping_rounds=120),
]

overfit_configs = [
    dict(group="overfit", max_depth=8, eta=0.12, num_boost_round=5000, min_child_weight=1, gamma=0, subsample=1.00, colsample_bytree=1.00, reg_lambda=2, reg_alpha=0.0, early_stopping_rounds=None),
    dict(group="overfit", max_depth=10, eta=0.10, num_boost_round=8000, min_child_weight=1, gamma=0, subsample=1.00, colsample_bytree=1.00, reg_lambda=1, reg_alpha=0.0, early_stopping_rounds=None),
    dict(group="overfit", max_depth=12, eta=0.08, num_boost_round=10000, min_child_weight=1, gamma=0, subsample=1.00, colsample_bytree=1.00, reg_lambda=0.5, reg_alpha=0.0, early_stopping_rounds=400),
    dict(group="overfit", max_depth=9, eta=0.15, num_boost_round=4000, min_child_weight=1, gamma=0, subsample=1.00, colsample_bytree=1.00, reg_lambda=0, reg_alpha=0.0, early_stopping_rounds=None),
]

all_configs = []
for i, c in enumerate(underfit_configs, 1):
    c = c.copy(); c['config_id'] = f'U{i}'; all_configs.append(c)
for i, c in enumerate(tuned_configs, 1):
    c = c.copy(); c['config_id'] = f'T{i}'; all_configs.append(c)
for i, c in enumerate(overfit_configs, 1):
    c = c.copy(); c['config_id'] = f'O{i}'; all_configs.append(c)

results = [best_row]
for cfg in all_configs:
    results.append(train_eval_one(cfg, config_id=cfg['config_id'], detx=DET_X_SWEEP))

all_results_df = pd.DataFrame(results)
all_results_df = all_results_df.sort_values('test_accuracy', ascending=False, na_position='last').reset_index(drop=True)
print('Total configs run (including BEST):', len(all_results_df))
all_results_df[['config_id','group','train_accuracy','test_accuracy','test_auc','test_logloss','alpha','rand_num_spikes','error']].head(20)


## Optional detX cross-check on representative configs (best, underfit, overfit)


In [ ]:
detx_rows = []

for label, selector in [
    ("BEST_DETX", all_results_df[all_results_df['config_id'] == 'BEST'].head(1)),
    ("UNDERFIT_DETX", all_results_df[all_results_df['group'] == 'underfit'].sort_values('test_accuracy', ascending=True).head(1)),
    ("OVERFIT_DETX", all_results_df[all_results_df['group'] == 'overfit'].sort_values('train_accuracy', ascending=False).head(1)),
]:
    if len(selector) == 0:
        continue
    r = selector.iloc[0]
    cfg = {
        'group': label,
        'max_depth': int(r.max_depth),
        'eta': float(r.eta),
        'subsample': float(r.subsample),
        'colsample_bytree': float(r.colsample_bytree),
        'min_child_weight': float(r.min_child_weight),
        'gamma': float(r.gamma),
        'reg_alpha': float(r.reg_alpha),
        'reg_lambda': float(r.reg_lambda),
        'num_boost_round': int(r.num_boost_round),
        'early_stopping_rounds': int(r.early_stopping_rounds) if pd.notna(r.early_stopping_rounds) else None,
    }
    detx_rows.append(train_eval_one(cfg, config_id=label, detx=True))

pd.DataFrame(detx_rows)[['config_id','group','alpha','rand_num_spikes','detX_val','error']] if detx_rows else pd.DataFrame()


## Plot results with labeled points and trap coloring


In [ ]:
plot_df = all_results_df.copy()
plot_df['has_traps'] = pd.to_numeric(plot_df['rand_num_spikes'], errors='coerce').fillna(0) > 0
best_idx = plot_df['test_accuracy'].idxmax() if plot_df['test_accuracy'].notna().any() else None
plot_df['is_best'] = False
if best_idx is not None:
    plot_df.loc[best_idx, 'is_best'] = True

# color coding required by checklist:
# a) best test accuracy, b) models with traps, c) models without traps
plot_df['color_group'] = np.where(plot_df['is_best'], 'best', np.where(plot_df['has_traps'], 'with_traps', 'without_traps'))
color_map = {'best':'gold', 'with_traps':'crimson', 'without_traps':'royalblue'}

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for grp, g in plot_df.groupby('color_group'):
    axes[0].scatter(g['alpha'], g['test_accuracy'], s=80, c=color_map[grp], label=grp, edgecolors='black', linewidths=0.3)
    axes[1].scatter(g['alpha'], g['train_accuracy'], s=80, c=color_map[grp], label=grp, edgecolors='black', linewidths=0.3)
    axes[2].scatter(g['rand_num_spikes'], g['test_accuracy'], s=80, c=color_map[grp], label=grp, edgecolors='black', linewidths=0.3)

axes[0].axvline(2.0, linestyle='--', color='black', alpha=0.8)
axes[1].axvline(2.0, linestyle='--', color='black', alpha=0.8)
axes[0].set_xlabel('alpha'); axes[0].set_ylabel('test accuracy'); axes[0].set_title('Test accuracy vs alpha')
axes[1].set_xlabel('alpha'); axes[1].set_ylabel('train accuracy'); axes[1].set_title('Train accuracy vs alpha')
axes[2].set_xlabel('rand_num_spikes'); axes[2].set_ylabel('test accuracy'); axes[2].set_title('Test accuracy vs traps')

for _, r in plot_df.iterrows():
    if pd.notna(r['alpha']) and pd.notna(r['test_accuracy']):
        axes[0].annotate(r['config_id'], (r['alpha'], r['test_accuracy']), fontsize=8)
    if pd.notna(r['alpha']) and pd.notna(r['train_accuracy']):
        axes[1].annotate(r['config_id'], (r['alpha'], r['train_accuracy']), fontsize=8)
    if pd.notna(r['rand_num_spikes']) and pd.notna(r['test_accuracy']):
        axes[2].annotate(r['config_id'], (r['rand_num_spikes'], r['test_accuracy']), fontsize=8)

for ax in axes:
    ax.legend(loc='best')

plt.tight_layout()
plt.show()

summary_cols = [
    'config_id','group','train_accuracy','test_accuracy','test_auc','test_logloss',
    'alpha','rand_num_spikes','max_depth','eta','subsample','colsample_bytree',
    'reg_lambda','reg_alpha','num_boost_round','best_iteration','error'
]
summary_df = plot_df[summary_cols].copy()
summary_df


## Guardrail checks


In [ ]:
alpha_nan_count = int(all_results_df['alpha'].isna().sum())
alpha_total = int(len(all_results_df))
print(f'alpha NaN count: {alpha_nan_count} / {alpha_total}')
if alpha_nan_count >= max(3, int(0.4 * alpha_total)):
    print('WARNING: Many alpha values are NaN. Verify convert() usage is exactly convert(bst, X_train, y_train, ...).')

failed = all_results_df[all_results_df['error'].astype(str).str.len() > 0]
print('Failed configs:', len(failed))
if len(failed):
    display(failed[['config_id','group','error']])


## Save results and plots to Drive checkpoint folder


In [ ]:
from pathlib import Path

output_filename = globals().get('OUTPUT_FILENAME', 'santander_hyperparameter_sweep_results.csv')
checkpoint_csv = globals().get('CHECKPOINT_RESULTS_CSV', Path('checkpoints') / 'santander_hparam_sweep' / output_filename)

preferred_local_dir = Path('notebooks')
local_dir = preferred_local_dir if preferred_local_dir.exists() else Path('.')
out_csv_local = local_dir / output_filename

out_csv_local.parent.mkdir(parents=True, exist_ok=True)
checkpoint_csv.parent.mkdir(parents=True, exist_ok=True)

all_results_df.to_csv(out_csv_local, index=False)
all_results_df.to_csv(checkpoint_csv, index=False)

plot_root = CHECKPOINT_ROOT if IN_COLAB else checkpoint_csv.parent
plot_root.mkdir(parents=True, exist_ok=True)

fig_test_path = plot_root / 'santander_acc_vs_alpha_test.png'
fig_train_path = plot_root / 'santander_acc_vs_alpha_train.png'
fig_trap_path = plot_root / 'santander_test_vs_spikes.png'

# Recreate lightweight save figures
plt.figure(figsize=(7,5))
for grp, g in all_results_df.assign(has_traps=pd.to_numeric(all_results_df['rand_num_spikes'], errors='coerce').fillna(0)>0).assign(is_best=False).groupby(lambda _: 'all'):
    pass

tmp = all_results_df.copy()
tmp['has_traps'] = pd.to_numeric(tmp['rand_num_spikes'], errors='coerce').fillna(0) > 0
tmp['is_best'] = False
if tmp['test_accuracy'].notna().any():
    tmp.loc[tmp['test_accuracy'].idxmax(), 'is_best'] = True
tmp['color_group'] = np.where(tmp['is_best'], 'best', np.where(tmp['has_traps'], 'with_traps', 'without_traps'))
color_map = {'best':'gold', 'with_traps':'crimson', 'without_traps':'royalblue'}

plt.figure(figsize=(7,5))
for grp, g in tmp.groupby('color_group'):
    plt.scatter(g['alpha'], g['test_accuracy'], c=color_map[grp], label=grp, s=70)
    for _, r in g.iterrows():
        if pd.notna(r['alpha']) and pd.notna(r['test_accuracy']):
            plt.annotate(r['config_id'], (r['alpha'], r['test_accuracy']), fontsize=7)
plt.axvline(2.0, linestyle='--', color='black'); plt.xlabel('alpha'); plt.ylabel('test accuracy'); plt.title('Santander test accuracy vs alpha'); plt.legend(); plt.tight_layout(); plt.savefig(fig_test_path, dpi=170); plt.close()

plt.figure(figsize=(7,5))
for grp, g in tmp.groupby('color_group'):
    plt.scatter(g['alpha'], g['train_accuracy'], c=color_map[grp], label=grp, s=70)
    for _, r in g.iterrows():
        if pd.notna(r['alpha']) and pd.notna(r['train_accuracy']):
            plt.annotate(r['config_id'], (r['alpha'], r['train_accuracy']), fontsize=7)
plt.axvline(2.0, linestyle='--', color='black'); plt.xlabel('alpha'); plt.ylabel('train accuracy'); plt.title('Santander train accuracy vs alpha'); plt.legend(); plt.tight_layout(); plt.savefig(fig_train_path, dpi=170); plt.close()

plt.figure(figsize=(7,5))
for grp, g in tmp.groupby('color_group'):
    plt.scatter(g['rand_num_spikes'], g['test_accuracy'], c=color_map[grp], label=grp, s=70)
    for _, r in g.iterrows():
        if pd.notna(r['rand_num_spikes']) and pd.notna(r['test_accuracy']):
            plt.annotate(r['config_id'], (r['rand_num_spikes'], r['test_accuracy']), fontsize=7)
plt.xlabel('rand_num_spikes'); plt.ylabel('test accuracy'); plt.title('Santander test accuracy vs traps'); plt.legend(); plt.tight_layout(); plt.savefig(fig_trap_path, dpi=170); plt.close()

print('Saved local copy:', out_csv_local.resolve())
print('Saved checkpoint CSV:', checkpoint_csv)
print('Saved plot:', fig_test_path)
print('Saved plot:', fig_train_path)
print('Saved plot:', fig_trap_path)

display(all_results_df[['config_id','group','train_accuracy','test_accuracy','alpha','rand_num_spikes','error']].head(30))
